# Topics 2 · Explore and name

1. **Explore**: interactive map of sample reviews (hover to read them), the same map by language, a tree of similar topics, keywords (topic shares per language and game: `eda/05_audience`). Topics are shown by their label once named, by keywords before.

The same theme in several languages gets the same label; analysis by label then counts them together.

In [ ]:
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.cluster.hierarchy import dendrogram, linkage

from src.viz.charts import use_cjk_font
from src.config import TOPIC_NOISE as NOISE
from src.config import TOPICS_DIR, game_names
from src.db import read, write
from src.text.topics import FIXED_LABELS, topic_names

# Charts are interactive (plotly). If one doesn't render, use fig.show(renderer="browser") instead of fig.show().
use_cjk_font()  # topic keywords can be Chinese

In [ ]:
LABELS = TOPICS_DIR / "labels.csv"
MANUAL = {}                 # quick fixes that win over labels.csv, e.g. {12: "Server connection problems"}
MAP_POINTS = 20_000         # reviews on the interactive map (more = slower in the browser)

In [ ]:
topics = read.topics()
sample = pd.read_parquet(TOPICS_DIR / "sample.parquet")
words = pd.read_parquet(TOPICS_DIR / "topic_words.parquet")
centroids = np.load(TOPICS_DIR / "centroids.npy")
K = len(centroids)
names = topic_names(topics)
print(f"{K} topics, {topics['label'].notna().sum()} named; sample of {len(sample):,} reviews")

## Explore

### Map of reviews

Each dot is a sample review, placed so that similar reviews are close. Hover to read it; click a topic in the legend to hide it, double-click to show only that topic. Grey = no clear topic. Numbers mark topic centers.

In [ ]:
shown = sample.dropna(subset=["x"]).sample(min(MAP_POINTS, sample["x"].notna().sum()), random_state=0)
shown = shown.assign(
    name=shown["topic"].map(names),
    text=shown["review"].fillna("").map(lambda s: "<br>".join(textwrap.wrap(" ".join(s.split())[:300], 70))),
    vote=shown["voted_up"].map({True: "👍", False: "👎"}),
)


def review_map(color, title, height, **options):
    """The shown sample reviews on the 2D map, colored by `color`; hover shows the review."""
    fig = px.scatter(
        shown, x="x", y="y", color=color, hover_name="name", render_mode="webgl", height=height, title=title,
        hover_data={"x": False, "y": False, "name": False, "text": True, "text_language": True, "game": True, "vote": True},
        **options,
    )
    fig.update_traces(marker_size=3)
    fig.update_layout(legend=dict(itemsizing="constant", title=None, font=dict(size=10)), plot_bgcolor="white")
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return fig


palette = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
fig = review_map(
    "name", "Sample reviews by topic", 850,
    category_orders={"name": [names[t] for t in sorted(shown["topic"].unique())]},
    color_discrete_sequence=palette, color_discrete_map={names[NOISE]: "#d9d9d9"},
)
centers = shown[shown["topic"] >= 0].groupby("topic")[["x", "y"]].median()
fig.add_scatter(x=centers["x"], y=centers["y"], text=centers.index.astype(str), mode="text",
                textfont=dict(size=10), showlegend=False, hoverinfo="skip")
fig.show()

Same map by language: mixed colors = a topic shared across languages, one-color islands = topics only one language writes about.

In [ ]:
fig = review_map("text_language", "Sample reviews by language", 750,
                 color_discrete_map={"en": "#2a78d6", "ru": "#eb6834", "zh": "#1baf7a"})
fig.update_traces(opacity=0.6)
fig.show()

### Topic tree

Topics joined early (on the left) have similar meaning; the same theme in another language usually sits right next to it. Useful for deciding which topics share a label.

In [ ]:
main_lang = sample[sample["topic"] >= 0].groupby("topic")["text_language"].agg(
    lambda s: f"{s.value_counts(normalize=True).index[0]} {s.value_counts(normalize=True).iloc[0]:.0%}"
)
leaf_labels = [f"{names[t][:55]}  ({main_lang.get(t, '')})" for t in range(K)]
fig, ax = plt.subplots(figsize=(11, 0.24 * K + 1))
dendrogram(linkage(centroids, method="average", metric="cosine"), orientation="left",
           labels=leaf_labels, ax=ax, leaf_font_size=8, color_threshold=0.15)
ax.set_xlabel("cosine distance between topic centers")
ax.spines[["top", "right", "left"]].set_visible(False)
plt.tight_layout()

### Keywords of the biggest topics

In [ ]:
biggest = topics[topics["topic_id"] >= 0].nlargest(16, "n_reviews")["topic_id"]
fig, axes = plt.subplots(4, 4, figsize=(15, 11))
for ax, t in zip(axes.flat, biggest):
    w = words[(words["topic"] == t) & (words["rank"] < 6)].iloc[::-1]
    ax.barh(w["word"], w["score"], color="#2a78d6")
    ax.set_title(textwrap.shorten(names[t], 40), loc="left", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_xticks([])
    ax.spines[["top", "right", "bottom"]].set_visible(False)
plt.tight_layout()

## Name

In [ ]:
if not LABELS.exists():
    template = topics[topics["topic_id"] >= 0].assign(label="")[["topic_id", "label", "keywords"]]
    template.to_csv(LABELS, index=False)
    print(f"created {LABELS} with {len(template)} topics: fill in the label column")
labels_file = pd.read_csv(LABELS, dtype={"label": str})
print(f"labels.csv: {labels_file['label'].notna().sum()} of {len(labels_file)} topics named")

### Save labels

In [ ]:
labels = labels_file.dropna(subset=["label"]).set_index("topic_id")["label"].str.strip().to_dict() | MANUAL | FIXED_LABELS
unknown = set(labels) - set(topics["topic_id"])
assert not unknown, f"labels for topics that are not in the database (made for another fit?): {sorted(unknown)}"
unnamed = sorted(set(topics["topic_id"]) - set(labels))
print(f"{len(topics)} topics, {len(unnamed)} without a name: {unnamed}")

write.set_topic_labels(labels)
topics = read.topics()
names = topic_names(topics)
topics["share"] = topics["n_reviews"] / topics["n_reviews"].sum()
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 100):
    display(topics.sort_values("n_reviews", ascending=False)[["topic_id", "label", "share", "keywords"]]
            .style.format({"share": "{:.1%}"}).hide(axis="index"))

Per label: topics with the same label (one theme in several languages) count together.

In [ ]:
by_label = topics.groupby("label").agg(topics=("topic_id", list), share=("share", "sum"))
by_label.sort_values("share", ascending=False).style.format({"share": "{:.1%}"})